# Test.csv 3-Model COMET Eval Only

- Purpose: evaluate pre-generated translations (`translations.csv`) in a COMET-compatible kernel.
- This notebook is separate from unsloth inference to avoid dependency conflicts.
- Expected input: `translations.csv` from translation notebook.


In [ ]:
# Runtime bootstrap for COMET-eval kernel (transformers<5.0)
import subprocess, sys

def pip_install(args, optional=False):
    cmd = [sys.executable, '-m', 'pip', 'install', '-q'] + list(args)
    print('running:', ' '.join(cmd))
    try:
        subprocess.check_call(cmd)
        return True
    except Exception as e:
        if optional:
            print(f'[WARN] optional install failed: {args} -> {type(e).__name__}: {e}')
            return False
        raise

pip_install(['pip','wheel','setuptools<82','Cython<3'])
pip_install(['pandas','numpy','tqdm','sacrebleu','rouge-score','bert-score','matplotlib','seaborn'])
pip_install(['huggingface_hub<1.0','transformers<5.0','--prefer-binary'])
pip_install(['PyYAML>=6.0.2','--prefer-binary','--only-binary=:all:'])
pip_install(['unbabel-comet==2.2.7','--prefer-binary'])
print('bootstrap install done. Restart kernel, then Run All.')


In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
import transformers
import huggingface_hub

print('transformers', transformers.__version__)
print('huggingface_hub', huggingface_hub.__version__)
print('cuda_available', torch.cuda.is_available())


In [ ]:
# Paths + config
def _find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for base in [start, *start.parents, Path('/workspace/scp_stage3_it'), Path('/workspace')]:
        if (base/'configs'/'prompts'/'translation_dynamic_5.yaml').exists():
            return base.resolve()
    return start

WORK_ROOT = Path(os.environ.get('WORK_ROOT','')).expanduser().resolve() if os.environ.get('WORK_ROOT','').strip() else _find_repo_root(Path.cwd())
OUTPUT_DIR = Path(os.environ.get('OUTPUT_DIR', str(WORK_ROOT/'outputs'/'testcsv_3model'))).expanduser().resolve()
TRANSLATIONS_CSV = Path(os.environ.get('TRANSLATIONS_CSV', str(OUTPUT_DIR/'translations.csv'))).expanduser().resolve()
METRICS = ['BLEU','chrF','TER','ROUGE-L','BERTScore-F1','COMET-ref']
COMET_MODEL_CANDIDATES = ['Unbabel/XCOMET-XXL','Unbabel/XCOMET-XL']
COMET_BATCH_SIZE = int(os.environ.get('COMET_BATCH_SIZE','8'))
COMET_NUM_WORKERS = int(os.environ.get('COMET_NUM_WORKERS','2'))
print('WORK_ROOT', WORK_ROOT)
print('OUTPUT_DIR', OUTPUT_DIR)
print('TRANSLATIONS_CSV', TRANSLATIONS_CSV)


In [ ]:
# Load translations
if not TRANSLATIONS_CSV.exists():
    raise FileNotFoundError(f'translations.csv not found: {TRANSLATIONS_CSV}')
translations_df = pd.read_csv(TRANSLATIONS_CSV, encoding='utf-8-sig').copy()
required_cols = {'row_id','model_name','source_text','reference_text','hypothesis','error'}
if not required_cols.issubset(set(translations_df.columns)):
    raise ValueError(f'Missing columns. required={sorted(required_cols)}, available={list(translations_df.columns)}')
valid_df = translations_df[translations_df['error'].fillna('').eq('') & translations_df['hypothesis'].fillna('').astype(str).str.strip().ne('') & translations_df['reference_text'].fillna('').astype(str).str.strip().ne('')].copy()
valid_df['row_id'] = pd.to_numeric(valid_df['row_id'], errors='coerce')
valid_df = valid_df.dropna(subset=['row_id']).copy()
valid_df['row_id'] = valid_df['row_id'].astype(int)
print('rows:', len(translations_df), 'valid_for_metrics:', len(valid_df))
display(valid_df.head(3))


In [ ]:
# Row-level metrics (incl. COMET)
score_rows=[]
if valid_df.empty:
    raise ValueError('No valid rows for metric computation.')

from sacrebleu.metrics import BLEU, CHRF, TER
bleu_metric = BLEU(effective_order=True, smooth_method='exp')
chrf_metric = CHRF(word_order=2)
ter_metric = TER()
if 'BLEU' in METRICS:
    for _,r in tqdm(valid_df.iterrows(), total=len(valid_df), desc='BLEU'):
        score_rows.append({'row_id':int(r['row_id']),'model_name':str(r['model_name']),'metric_name':'BLEU','metric_value':float(bleu_metric.sentence_score(str(r['hypothesis']), [str(r['reference_text'])]).score)})
if 'chrF' in METRICS:
    for _,r in tqdm(valid_df.iterrows(), total=len(valid_df), desc='chrF'):
        score_rows.append({'row_id':int(r['row_id']),'model_name':str(r['model_name']),'metric_name':'chrF','metric_value':float(chrf_metric.sentence_score(str(r['hypothesis']), [str(r['reference_text'])]).score)})
if 'TER' in METRICS:
    for _,r in tqdm(valid_df.iterrows(), total=len(valid_df), desc='TER'):
        score_rows.append({'row_id':int(r['row_id']),'model_name':str(r['model_name']),'metric_name':'TER','metric_value':float(ter_metric.sentence_score(str(r['hypothesis']), [str(r['reference_text'])]).score)})
if 'ROUGE-L' in METRICS:
    from rouge_score import rouge_scorer
    rouge = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)
    for _,r in tqdm(valid_df.iterrows(), total=len(valid_df), desc='ROUGE-L'):
        v = rouge.score(str(r['reference_text']), str(r['hypothesis']))['rougeL'].fmeasure
        score_rows.append({'row_id':int(r['row_id']),'model_name':str(r['model_name']),'metric_name':'ROUGE-L','metric_value':float(v)})
if 'BERTScore-F1' in METRICS:
    try:
        from bert_score import score as bertscore_score
        for model_name, g in tqdm(valid_df.groupby('model_name'), total=valid_df['model_name'].nunique(), desc='BERTScore'):
            cands = g['hypothesis'].astype(str).tolist(); refs = g['reference_text'].astype(str).tolist()
            _, _, f1 = bertscore_score(cands, refs, lang='ko', rescale_with_baseline=False, verbose=False)
            for rid,val in zip(g['row_id'].astype(int).tolist(), f1.tolist()):
                score_rows.append({'row_id':int(rid),'model_name':str(model_name),'metric_name':'BERTScore-F1','metric_value':float(val)})
    except Exception as e:
        print('[WARN] BERTScore-F1 skipped:', e)

if 'COMET-ref' in METRICS:
    from comet import download_model, load_from_checkpoint
    try:
        from huggingface_hub import HfApi
        _api = HfApi(); _tok = os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_HUB_TOKEN')
        for candidate in COMET_MODEL_CANDIDATES:
            try:
                _api.model_info(candidate, token=_tok)
            except Exception as _e:
                print(f'[COMET][WARN] HF access check for {candidate}: {type(_e).__name__}: {_e}')
    except Exception as _e:
        print(f'[COMET][WARN] huggingface_hub precheck skipped: {type(_e).__name__}: {_e}')
    selected=None; model=None; last_err=None
    for candidate in COMET_MODEL_CANDIDATES:
        try:
            print(f'[COMET] trying model: {candidate}')
            mpath = download_model(candidate)
            model = load_from_checkpoint(mpath)
            selected = candidate
            break
        except Exception as e:
            last_err=e
            print(f'[COMET][WARN] failed to load {candidate}: {type(e).__name__}: {e}')
    if model is None:
        raise RuntimeError('Unable to load any XCOMET model. Check HF terms/auth for gated repos.') from last_err
    print('[COMET] selected model:', selected)
    gpus = 1 if torch.cuda.is_available() else 0
    for model_name, g in tqdm(valid_df.groupby('model_name'), total=valid_df['model_name'].nunique(), desc='COMET-ref'):
        data = [{'src':s,'mt':h,'ref':r} for s,h,r in zip(g['source_text'].astype(str), g['hypothesis'].astype(str), g['reference_text'].astype(str))]
        pred = model.predict(data, batch_size=int(COMET_BATCH_SIZE), gpus=gpus, progress_bar=False, num_workers=int(COMET_NUM_WORKERS))
        for rid,val in zip(g['row_id'].astype(int).tolist(), pred.scores):
            score_rows.append({'row_id':int(rid),'model_name':str(model_name),'metric_name':'COMET-ref','metric_value':float(val)})

scores_df = pd.DataFrame(score_rows, columns=['row_id','model_name','metric_name','metric_value']).sort_values(['row_id','model_name','metric_name']).reset_index(drop=True)
if 'COMET-ref' in METRICS:
    expected=len(valid_df); actual=int((scores_df['metric_name']=='COMET-ref').sum())
    if actual != expected:
        raise RuntimeError(f'COMET-ref incomplete. expected_rows={expected}, actual_rows={actual}')
scores_path = OUTPUT_DIR/'row_level_scores.csv'
scores_df.to_csv(scores_path, index=False, encoding='utf-8-sig')
print('saved:', scores_path, 'rows=', len(scores_df))
display(scores_df.head(10))


In [ ]:
# Summary + composite
if scores_df.empty:
    raise ValueError('No score rows')
higher_is_better={'BLEU':True,'chrF':True,'TER':False,'ROUGE-L':True,'BERTScore-F1':True,'COMET-ref':True}
summary=(scores_df.groupby(['model_name','metric_name'], as_index=False).agg(mean=('metric_value','mean'), std=('metric_value','std')))
rank_rows=[]
for metric_name,g in summary.groupby('metric_name'):
    asc = not higher_is_better.get(metric_name, True)
    g2 = g.sort_values('mean', ascending=asc).reset_index(drop=True)
    g2['rank']=np.arange(1,len(g2)+1)
    rank_rows.append(g2)
summary=pd.concat(rank_rows, axis=0, ignore_index=True)
norm_rows=[]
for metric_name,g in summary.groupby('metric_name'):
    vals=g['mean'].astype(float).values
    vmin,vmax=float(np.min(vals)),float(np.max(vals))
    norm=np.ones_like(vals) if (vmax-vmin)<1e-12 else (vals-vmin)/(vmax-vmin)
    if not higher_is_better.get(metric_name, True): norm=1.0-norm
    for model_name,n in zip(g['model_name'].tolist(), norm.tolist()):
        norm_rows.append({'model_name':model_name,'metric_name':metric_name,'normalized':float(n)})
norm_df=pd.DataFrame(norm_rows)
composite=norm_df.groupby('model_name', as_index=False)['normalized'].mean().rename(columns={'normalized':'composite_score'})
composite['composite_rank']=composite['composite_score'].rank(ascending=False, method='min').astype(int)
model_summary=summary.merge(composite, on='model_name', how='left')
model_summary=model_summary[['model_name','metric_name','mean','std','rank','composite_score','composite_rank']]
summary_path=OUTPUT_DIR/'model_summary.csv'
model_summary.to_csv(summary_path, index=False, encoding='utf-8-sig')
print('saved:', summary_path)
display(model_summary.sort_values(['metric_name','rank','model_name']))
display(composite.sort_values('composite_score', ascending=False))
